In [1]:
"""
=============================================================================
NOTEBOOK 01 COMPLETO: BUILD STOCK-LEVEL PANEL (CONGRESO + MERCADO)
=============================================================================
Proyecto: Detección de Insider Trading en el Congreso de EE.UU.
Maestría en Economía - UdeSA

Este notebook construye el panel COMPLETO a nivel (ACCIÓN, MES) incluyendo:
- Variables de actividad de congresistas
- Variables de mercado (ya están en tu archivo)
- Variable objetivo: retorno del mes siguiente

OUTPUT: panel_final_stock_month.parquet
=============================================================================
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import os

# =============================================================================
# 0. CONFIGURACIÓN
# =============================================================================

os.chdir('C:/Users/sebib/Documents/GitHub/US_Congress')
print(f"Directorio de trabajo: {os.getcwd()}")

INPUT_TRADES = 'data/outputs/congress_trades_with_committees.parquet'
OUTPUT_DIR = 'data/prediction_bases'
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = '2012-01-01'
END_DATE = '2024-12-31'

INFO_COMMITTEES = [
    'Armed Services', 'Financial Services', 'Energy and Commerce',
    'Intelligence', 'Select Committee on Intelligence',
    'Ways and Means', 'Appropriations', 'Health Education Labor and Pensions',
    'Banking, Housing and Urban Affairs', 'Finance',
    'Judiciary', 'Commerce, Science and Transportation'
]

print("="*70)
print("NOTEBOOK 01: BUILD COMPLETE STOCK-LEVEL PANEL")
print("="*70)

# =============================================================================
# 1. CARGAR DATOS
# =============================================================================
print("\n[1] CARGANDO DATOS...")

df = pd.read_parquet(INPUT_TRADES)
print(f"    Trades cargados: {len(df):,}")
print(f"    Columnas: {len(df.columns)}")

# =============================================================================
# 2. PREPARACIÓN
# =============================================================================
print("\n[2] PREPARACIÓN DE DATOS...")

# Fechas
df['trade_date'] = pd.to_datetime(df['Traded'])
df['trade_month'] = df['trade_date'].dt.to_period('M')

# Filtrar período
df = df[(df['trade_date'] >= START_DATE) & (df['trade_date'] <= END_DATE)]
print(f"    Trades en período: {len(df):,}")
print(f"    Acciones únicas: {df['Ticker_Clean'].nunique():,}")

# =============================================================================
# 3. CREAR FEATURES DE CONGRESO A NIVEL TRADE
# =============================================================================
print("\n[3] CREANDO FEATURES DE CONGRESO...")

# --- Dirección ---
df['is_buy'] = df['Transaction'].str.lower().str.contains('purchase|buy', na=False).astype(int)
df['is_sell'] = df['Transaction'].str.lower().str.contains('sale|sell', na=False).astype(int)
print(f"    Compras: {df['is_buy'].sum():,}, Ventas: {df['is_sell'].sum():,}")

# --- Monto ---
size_map = {
    '$1,001 - $15,000': 8000,
    '$15,001 - $50,000': 32500,
    '$50,001 - $100,000': 75000,
    '$100,001 - $250,000': 175000,
    '$250,001 - $500,000': 375000,
    '$500,001 - $1,000,000': 750000,
    '$1,000,001 - $5,000,000': 3000000,
    'Over $5,000,000': 7500000,
}
df['amount_proxy'] = df['Trade_Size_USD'].map(size_map).fillna(8000)
df['is_large_trade'] = (df['amount_proxy'] >= 100000).astype(int)

# --- Timing ---
df['filed_date'] = pd.to_datetime(df['Filed'])
df['disclosure_delay'] = (df['filed_date'] - df['trade_date']).dt.days.clip(lower=0, upper=365)
df['long_delay'] = (df['disclosure_delay'] > 30).astype(int)
df['end_of_month'] = (df['trade_date'].dt.daysinmonth - df['trade_date'].dt.day <= 5).astype(int)

# --- Comités ---
def is_info_committee(x):
    if pd.isna(x): return 0
    return int(any(ic.lower() in str(x).lower() for ic in INFO_COMMITTEES))

df['is_info_committee'] = df['committee_name'].apply(is_info_committee)
df['is_chair'] = df['committee_role'].fillna('').str.lower().str.contains('chair|ranking').astype(int)
print(f"    Info committee: {df['is_info_committee'].mean()*100:.1f}%")

# --- Poder ---
df['years_in_position'] = pd.to_numeric(df['Years in position'], errors='coerce').fillna(0)
df['is_senior'] = (df['years_in_position'] >= 10).astype(int)
df['net_worth_num'] = pd.to_numeric(df['Net worth'], errors='coerce').fillna(0)
df['is_senator'] = (df['chamber'] == 2).astype(int) if 'chamber' in df.columns else 0
df['power_index'] = df['is_chair'] + df['is_senator'] + df['is_senior'] + df['is_info_committee']

# --- Comportamiento ---
trader_counts = df.groupby('Name')['trade_date'].count()
frequent_traders = trader_counts[trader_counts >= trader_counts.quantile(0.75)].index
df['frequent_trader'] = df['Name'].isin(frequent_traders).astype(int)

df = df.sort_values(['Name', 'Ticker_Clean', 'trade_date'])
df['first_time'] = (~df.duplicated(subset=['Name', 'Ticker_Clean'], keep='first')).astype(int)

# --- Coordinación ---
daily_traders = df.groupby(['trade_date', 'Ticker_Clean'])['Name'].nunique().reset_index()
daily_traders.columns = ['trade_date', 'Ticker_Clean', 'n_traders_same_day']
df = df.merge(daily_traders, on=['trade_date', 'Ticker_Clean'], how='left')
df['coordinated'] = (df['n_traders_same_day'] >= 2).astype(int)

print(f"    Coordinados: {df['coordinated'].mean()*100:.1f}%")

# =============================================================================
# 4. IDENTIFICAR VARIABLES DE MERCADO DISPONIBLES
# =============================================================================
print("\n[4] IDENTIFICANDO VARIABLES DE MERCADO...")

# Variables de mercado que ya tenés en el archivo
MARKET_VARS = [
    # Retornos
    'return_t', 'excess_return',
    # Momentum
    'momentum_5d', 'momentum_20d', 'momentum_60d', 'momentum_252d',
    # Volatilidad
    'realized_vol_30d', 'parkinson_vol_30d', 'realized_vol_60d', 
    'vol_of_vol_60d', 'realized_vol_252d',
    # Volumen/Liquidez
    'volume_ratio_30d', 'abnormal_volume_30d', 'amihud_illiq_20d',
    'roll_spread_30d', 'hl_spread_20d', 'zero_volume_days_30d',
    # Riesgo
    'beta_252d', 'r2_market_252d',
    # Fama-French
    'alpha_ff3_252d', 'beta_mkt_ff3_252d', 'beta_smb_ff3_252d', 
    'beta_hml_ff3_252d', 'r2_ff3_252d',
    # Fundamentales
    'market_cap', 'price', 'book_value', 'price_to_book', 'ev_to_ebitda',
    # CARs (estos son outcomes, no features)
    'car_raw_30d', 'car_capm_30d', 'car_ff3_30d',
    'car_raw_60d', 'car_capm_60d', 'car_ff3_60d',
    'car_raw_90d', 'car_capm_90d', 'car_ff3_90d',
]

# Verificar cuáles existen
available_market = [v for v in MARKET_VARS if v in df.columns]
print(f"    Variables de mercado disponibles: {len(available_market)}")

# =============================================================================
# 5. COLAPSAR A NIVEL (ACCIÓN, MES)
# =============================================================================
print("\n[5] COLAPSANDO A NIVEL (ACCIÓN, MES)...")

# --- 5.1 Variables de Congreso ---
print("    5.1 Colapsando variables de congreso...")

unique_politicians = df.groupby(['Ticker_Clean', 'trade_month'])['Name'].nunique().reset_index()
unique_politicians.columns = ['ticker', 'month', 'cong_unique_politicians']

agg_cong = df.groupby(['Ticker_Clean', 'trade_month']).agg(
    # Conteos
    cong_total_trades=('is_buy', 'count'),
    cong_buy_count=('is_buy', 'sum'),
    cong_sell_count=('is_sell', 'sum'),
    cong_total_amount=('amount_proxy', 'sum'),
    cong_large_trades=('is_large_trade', 'sum'),
    # Timing
    cong_avg_disclosure_delay=('disclosure_delay', 'mean'),
    cong_long_delay_trades=('long_delay', 'sum'),
    cong_end_of_month_trades=('end_of_month', 'sum'),
    # Comités
    cong_info_committee_trades=('is_info_committee', 'sum'),
    cong_chair_trades=('is_chair', 'sum'),
    # Poder
    cong_senior_trades=('is_senior', 'sum'),
    cong_senator_trades=('is_senator', 'sum'),
    cong_avg_power_index=('power_index', 'mean'),
    cong_max_power_index=('power_index', 'max'),
    cong_avg_seniority=('years_in_position', 'mean'),
    # Partido
    cong_dem_trades=('is_democrat', 'sum'),
    cong_rep_trades=('is_republican', 'sum'),
    # Comportamiento
    cong_frequent_trader_trades=('frequent_trader', 'sum'),
    cong_first_time_trades=('first_time', 'sum'),
    # Coordinación
    cong_coordinated_trades=('coordinated', 'sum'),
    cong_max_traders_same_day=('n_traders_same_day', 'max'),
).reset_index()

agg_cong.columns = ['ticker', 'month'] + list(agg_cong.columns[2:])
agg_cong = agg_cong.merge(unique_politicians, on=['ticker', 'month'], how='left')

# --- 5.2 Variables de Mercado (promedios del mes) ---
print("    5.2 Colapsando variables de mercado...")

# Para mercado, tomamos el ÚLTIMO valor del mes (más reciente) o promedio
agg_dict_market = {}
for var in available_market:
    if 'car_' in var:
        # Para CARs, tomar el promedio (son outcomes)
        agg_dict_market[f'mkt_{var}'] = (var, 'mean')
    elif var in ['market_cap', 'price', 'book_value']:
        # Para fundamentales, tomar el último
        agg_dict_market[f'mkt_{var}'] = (var, 'last')
    else:
        # Para el resto, tomar el promedio del mes
        agg_dict_market[f'mkt_{var}'] = (var, 'mean')

agg_market = df.groupby(['Ticker_Clean', 'trade_month']).agg(**agg_dict_market).reset_index()
agg_market.columns = ['ticker', 'month'] + list(agg_market.columns[2:])

# --- 5.3 Mergear Congreso + Mercado ---
print("    5.3 Mergeando...")
panel = agg_cong.merge(agg_market, on=['ticker', 'month'], how='left')

print(f"    Panel base: {len(panel):,} observaciones")

# =============================================================================
# 6. CREAR VARIABLES DERIVADAS
# =============================================================================
print("\n[6] CREANDO VARIABLES DERIVADAS...")

# --- Variables de Congreso derivadas ---
panel['cong_net'] = panel['cong_buy_count'] - panel['cong_sell_count']
panel['cong_buy_ratio'] = panel['cong_buy_count'] / panel['cong_total_trades']
panel['cong_csi'] = panel['cong_net'] / panel['cong_total_trades']

# Ratios
total = panel['cong_total_trades']
panel['cong_info_ratio'] = panel['cong_info_committee_trades'] / total
panel['cong_chair_ratio'] = panel['cong_chair_trades'] / total
panel['cong_senior_ratio'] = panel['cong_senior_trades'] / total
panel['cong_senator_ratio'] = panel['cong_senator_trades'] / total
panel['cong_dem_ratio'] = panel['cong_dem_trades'] / total
panel['cong_coordinated_ratio'] = panel['cong_coordinated_trades'] / total
panel['cong_first_time_ratio'] = panel['cong_first_time_trades'] / total
panel['cong_large_ratio'] = panel['cong_large_trades'] / total
panel['cong_long_delay_ratio'] = panel['cong_long_delay_trades'] / total
panel['cong_intensity'] = panel['cong_total_trades'] / panel['cong_unique_politicians']

# Binarias
panel['cong_consensus_buy'] = (panel['cong_buy_ratio'] > 0.7).astype(int)
panel['cong_consensus_sell'] = (panel['cong_buy_ratio'] < 0.3).astype(int)
panel['cong_multiple_politicians'] = (panel['cong_unique_politicians'] > 1).astype(int)

# Compuestas
panel['cong_smart_money'] = ((panel['cong_net'] > 0) & 
                              (panel['cong_info_committee_trades'] > 0) & 
                              (panel['cong_chair_trades'] > 0)).astype(int)

panel['cong_strong_buy'] = ((panel['cong_csi'] > 0.5) & 
                             (panel['cong_unique_politicians'] >= 2)).astype(int)

panel['cong_strong_sell'] = ((panel['cong_csi'] < -0.5) & 
                              (panel['cong_unique_politicians'] >= 2)).astype(int)

# =============================================================================
# 7. CREAR VARIABLE OBJETIVO (RETORNO FUTURO)
# =============================================================================
print("\n[7] CREANDO VARIABLE OBJETIVO (RETORNO FUTURO)...")

# Ordenar por ticker y mes
panel = panel.sort_values(['ticker', 'month'])

# Crear retorno del mes siguiente para cada acción
# Usamos el retorno promedio del mes actual y lo shifteamos
if 'mkt_return_t' in panel.columns:
    panel['ret_future_1m'] = panel.groupby('ticker')['mkt_return_t'].shift(-1)
    print(f"    Variable objetivo creada: ret_future_1m")
    print(f"    Observaciones con ret_future: {panel['ret_future_1m'].notna().sum():,}")

# También crear retornos futuros usando CARs si están disponibles
if 'mkt_car_raw_30d' in panel.columns:
    # El CAR ya es el retorno futuro! Está calculado como retorno post-trade
    panel['ret_future_car30'] = panel['mkt_car_raw_30d']
    print(f"    Variable alternativa: ret_future_car30 (CAR 30 días)")

if 'mkt_car_ff3_30d' in panel.columns:
    panel['ret_future_car30_ff3'] = panel['mkt_car_ff3_30d']
    print(f"    Variable alternativa: ret_future_car30_ff3 (CAR FF3 30 días)")

# =============================================================================
# 8. DEFINIR CONJUNTOS DE FEATURES
# =============================================================================
print("\n[8] DEFINIENDO CONJUNTOS DE FEATURES...")

# Features de MERCADO (modelo base)
FEATURES_MARKET = [col for col in panel.columns if col.startswith('mkt_') 
                   and 'car_' not in col]  # Excluir CARs (son outcomes)

# Features de CONGRESO
FEATURES_CONGRESS = [col for col in panel.columns if col.startswith('cong_')]

# Features COMPLETAS
FEATURES_ALL = FEATURES_MARKET + FEATURES_CONGRESS

print(f"    Features de mercado: {len(FEATURES_MARKET)}")
print(f"    Features de congreso: {len(FEATURES_CONGRESS)}")
print(f"    Features totales: {len(FEATURES_ALL)}")

# Guardar listas de features
features_dict = {
    'market': FEATURES_MARKET,
    'congress': FEATURES_CONGRESS,
    'all': FEATURES_ALL
}

# =============================================================================
# 9. LIMPIAR Y GUARDAR
# =============================================================================
print("\n[9] LIMPIANDO Y GUARDANDO...")

# Eliminar filas sin variable objetivo
panel_clean = panel.dropna(subset=['ret_future_1m'])
print(f"    Observaciones finales: {len(panel_clean):,}")

# Guardar panel completo
output_path = os.path.join(OUTPUT_DIR, 'panel_final_stock_month.parquet')
panel_clean.to_parquet(output_path, index=False)
print(f"    Guardado: {output_path}")

# Guardar CSV para inspección
csv_path = os.path.join(OUTPUT_DIR, 'panel_final_stock_month.csv')
panel_clean.to_csv(csv_path, index=False)
print(f"    Guardado: {csv_path}")

# Guardar diccionario de features
import json
features_path = os.path.join(OUTPUT_DIR, 'feature_sets.json')
with open(features_path, 'w') as f:
    json.dump(features_dict, f, indent=2)
print(f"    Guardado: {features_path}")

# =============================================================================
# 10. RESUMEN FINAL
# =============================================================================
print("\n" + "="*70)
print("RESUMEN - PANEL COMPLETO PARA MODELADO")
print("="*70)

print(f"""
DIMENSIONES FINALES:
  Observaciones:     {len(panel_clean):,}
  Acciones únicas:   {panel_clean['ticker'].nunique():,}
  Meses:             {panel_clean['month'].nunique()}
  Variables totales: {len(panel_clean.columns)}

FEATURES POR CATEGORÍA:
  Mercado (modelo base):     {len(FEATURES_MARKET)} variables
  Congreso (valor agregado): {len(FEATURES_CONGRESS)} variables

VARIABLES OBJETIVO DISPONIBLES:
  ret_future_1m:      Retorno del mes siguiente
  ret_future_car30:   CAR 30 días (si disponible)

DISEÑO EXPERIMENTAL:
  Modelo Base:     ret_future ~ f(FEATURES_MARKET)
  Modelo Ampliado: ret_future ~ f(FEATURES_MARKET + FEATURES_CONGRESS)
  
  Si R²_ampliado > R²_base → Los trades de congresistas tienen información

ARCHIVOS GENERADOS:
  - panel_final_stock_month.parquet
  - panel_final_stock_month.csv
  - feature_sets.json
""")

print("="*70)
print("✅ PANEL COMPLETO LISTO PARA MODELADO")
print("="*70)

# =============================================================================
# 11. ESTADÍSTICAS DESCRIPTIVAS
# =============================================================================
print("\n📊 ESTADÍSTICAS DE VARIABLES CLAVE:")

# Variable objetivo
print("\n--- Variable Objetivo ---")
print(panel_clean['ret_future_1m'].describe())

# Features de congreso clave
print("\n--- Features de Congreso ---")
cong_key = ['cong_net', 'cong_csi', 'cong_unique_politicians', 'cong_info_ratio']
print(panel_clean[cong_key].describe().round(3))

# Features de mercado clave
print("\n--- Features de Mercado ---")
mkt_key = [c for c in ['mkt_return_t', 'mkt_momentum_20d', 'mkt_realized_vol_30d', 'mkt_market_cap'] 
           if c in panel_clean.columns]
if mkt_key:
    print(panel_clean[mkt_key].describe().round(3))

# Correlaciones con variable objetivo
print("\n--- Top 10 Correlaciones con ret_future_1m ---")
numeric_cols = panel_clean.select_dtypes(include=[np.number]).columns
corr_with_target = panel_clean[numeric_cols].corr()['ret_future_1m'].drop('ret_future_1m')
top_corr = corr_with_target.abs().sort_values(ascending=False).head(10)
for var in top_corr.index:
    print(f"  {var}: {corr_with_target[var]:.4f}")

Directorio de trabajo: C:\Users\sebib\Documents\GitHub\US_Congress
NOTEBOOK 01: BUILD COMPLETE STOCK-LEVEL PANEL

[1] CARGANDO DATOS...
    Trades cargados: 99,609
    Columnas: 97

[2] PREPARACIÓN DE DATOS...
    Trades en período: 86,624
    Acciones únicas: 4,440

[3] CREANDO FEATURES DE CONGRESO...
    Compras: 43,826, Ventas: 42,388
    Info committee: 42.0%
    Coordinados: 5.9%

[4] IDENTIFICANDO VARIABLES DE MERCADO...
    Variables de mercado disponibles: 38

[5] COLAPSANDO A NIVEL (ACCIÓN, MES)...
    5.1 Colapsando variables de congreso...
    5.2 Colapsando variables de mercado...
    5.3 Mergeando...
    Panel base: 43,381 observaciones

[6] CREANDO VARIABLES DERIVADAS...

[7] CREANDO VARIABLE OBJETIVO (RETORNO FUTURO)...
    Variable objetivo creada: ret_future_1m
    Observaciones con ret_future: 32,169
    Variable alternativa: ret_future_car30 (CAR 30 días)
    Variable alternativa: ret_future_car30_ff3 (CAR FF3 30 días)

[8] DEFINIENDO CONJUNTOS DE FEATURES...
    Fea